In [7]:
import torch
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from torch import Tensor
from torch.nn import Sequential, Linear, ReLU, CrossEntropyLoss, Softplus, PReLU
from torch.utils.data import DataLoader
from torch.optim import Adam

In [8]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))



PyTorch: 2.14.1+cu126
CUDA disponível: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [9]:
dataset = load_iris()
dataset.data.shape
print(dataset.target_names)
print(dataset.feature_names)



['setosa' 'versicolor' 'virginica']
['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']


In [10]:
x = dataset.data
y = dataset.target

X_train,X_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42)

In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_normalizado = scaler.fit_transform(X_train)
X_test_normalizado = scaler.transform(X_test)

In [12]:
from sklearn.decomposition import PCA

N_components = 3
                                      
pca = PCA(n_components=N_components)

X_train_pca = pca.fit_transform(
    X_train_normalizado
)

X_test_pca = pca.transform(
    X_test_normalizado
)

In [13]:
input_shape = len(dataset.feature_names)
output_shape = len(dataset.target_names)

model = Sequential(
    Linear(N_components,4),
    ReLU(),
    Linear(4,8),
    ReLU(),
    Linear(8,output_shape)
)
model.to("cuda")
print(model)



Sequential(
  (0): Linear(in_features=3, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=8, bias=True)
  (3): ReLU()
  (4): Linear(in_features=8, out_features=3, bias=True)
)


In [14]:
optmizer = Adam(model.parameters(), lr= 0.001)
criterion = CrossEntropyLoss().to("cuda")
max_epochs = 1000
batch = 8

In [15]:
total_batchs = len(X_train) // batch
last_batch = len(X_train) % batch
has_last_b = last_batch > 0

print(total_batchs)
print(last_batch)



15
0


In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_tensor = torch.tensor(
    X_train_pca,
    dtype=torch.float32
).to(device)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.long
).to(device)

X_test_tensor = torch.tensor(
    X_test_pca,
    dtype=torch.float32
).to(device)



In [17]:
def train_model():

    model.train()

    for i_epochs in range(max_epochs):

        for i_batchs in range(total_batchs):

            inicio = i_batchs * batch
            fim = (i_batchs + 1) * batch

            batch_x = X_train_tensor[inicio:fim]
            batch_y = y_train_tensor[inicio:fim]

            optmizer.zero_grad()

            output = model(batch_x)
            loss = criterion(output, batch_y)

            loss.backward()
            optmizer.step()



In [18]:
def eval_model():

    model.eval()

    with torch.no_grad():

        output = model(X_test_tensor)

        y_pred = torch.argmax(output, dim=1)

    y_pred = y_pred.cpu().numpy()

    print(f"Acuraçy Score: {(accuracy_score(y_test, y_pred)*100):.2f}%")
    print(f"F1 Score:  {(f1_score(y_test, y_pred,average="macro")*100):.2f}%")
    print(f"Confusion Matrix: \n{confusion_matrix(y_test,y_pred)}")



In [19]:
train_model()
eval_model()

Acuraçy Score: 100.00%
F1 Score:  100.00%
Confusion Matrix: 
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]
